# NLA-KTH — RL Phase (GRPO) + Layer Ablation

**What this notebook does (end-to-end, ~4-5 hours on Kaggle 2x T4):**
1. Install dependencies and clone the nla-kth repo
2. Load activations + summaries from the `nla-data` dataset
3. Train the reconstructor (8 epochs)
4. Train the verbalizer warm-start (3 epochs)
5. Run GRPO RL on the verbalizer (200 steps)
6. Evaluate FVE **before** and **after** RL
7. Run layer ablation at layers {8, 12, 16, 20}
8. Save all results to `/kaggle/working/`

**Input dataset required:** `nla-data` (activations.npy, metadata.jsonl, summaries.jsonl)

Add it via: Notebook → Add Input → Datasets → search `mohamedibrahim191026/nla-data`

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.stdout

print(run('pip install -q peft transformers accelerate bitsandbytes datasets tqdm'))
print(run('git clone https://github.com/mohamedibrahim26/nla-kth.git /kaggle/working/nla-kth'))
sys.path.insert(0, '/kaggle/working/nla-kth/src')

In [ ]:
# ── 1. Load data from nla-data dataset ───────────────────────────────────────
import numpy as np, json, os, pickle

DATA_DIR  = '/kaggle/working/data'
INPUT_DIR = '/kaggle/input/nla-data'
os.makedirs(DATA_DIR, exist_ok=True)

# Load activations
acts = np.load(f'{INPUT_DIR}/activations.npy')          # (N, D)
print(f'Activations: {acts.shape}  dtype={acts.dtype}')

# Load metadata
meta = [json.loads(l) for l in open(f'{INPUT_DIR}/metadata.jsonl')]
print(f'Metadata records: {len(meta)}')

# Load teacher summaries
sums = [json.loads(l) for l in open(f'{INPUT_DIR}/summaries.jsonl')]
print(f'Summaries: {len(sums)}')

# Build paired dataset: (activation_idx, summary_text)
# summaries.jsonl has fields: idx (into activations), summary
paired = [(s['idx'], s['summary']) for s in sums if 'summary' in s and 'idx' in s]
print(f'Paired samples: {len(paired)}')

# Save in format train_reconstructor.py and train_verbalizer.py expect
np.save(f'{DATA_DIR}/activations.npy', acts)
with open(f'{DATA_DIR}/metadata.jsonl', 'w') as f:
    for m in meta: f.write(json.dumps(m) + '\n')
with open(f'{DATA_DIR}/summaries.jsonl', 'w') as f:
    for s in sums: f.write(json.dumps(s) + '\n')

print('Data ready in', DATA_DIR)

In [ ]:
# ── 2. Train Reconstructor (English → activation) ────────────────────────────
# ~60-90 minutes on 2x T4
print('=== STEP 2: Training Reconstructor ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/train_reconstructor.py \
    --data_dir {DATA_DIR} \
    --mode lora \
    --epochs 8 \
    --batch_size 16 \
    --lr 2e-4 \
    --lora_r 32
''')
print(out[-3000:])

In [ ]:
# Check reconstructor checkpoint saved
import os
for f in sorted(os.listdir(DATA_DIR)):
    size = os.path.getsize(f'{DATA_DIR}/{f}') if os.path.isfile(f'{DATA_DIR}/{f}') else '-'
    print(f'{f:40s}  {size}')

In [ ]:
# ── 3. Train Verbalizer warm-start (activation → English) ────────────────────
# ~30-45 minutes on 2x T4
print('=== STEP 3: Training Verbalizer (warm-start) ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/train_verbalizer.py \
    --data_dir {DATA_DIR} \
    --epochs 3 \
    --batch_size 4 \
    --lr 2e-4
''')
print(out[-3000:])

In [ ]:
# ── 4. Evaluate FVE BEFORE RL ────────────────────────────────────────────────
print('=== STEP 4: Evaluating warm-start FVE (before RL) ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/evaluate_nla.py \
    --data_dir {DATA_DIR} \
    --verbalizer_pt   {DATA_DIR}/verbalizer.pt \
    --verbalizer_lora {DATA_DIR}/verbalizer_lora
''')
print(out[-3000:])

# Copy results
import shutil
if os.path.exists(f'{DATA_DIR}/nla_results.json'):
    shutil.copy(f'{DATA_DIR}/nla_results.json', '/kaggle/working/nla_results_warmstart.json')
    import json
    r = json.load(open('/kaggle/working/nla_results_warmstart.json'))
    print('\n=== WARM-START FVE SUMMARY ===')
    for k, v in r.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')

In [ ]:
# ── 5. Run GRPO RL phase ──────────────────────────────────────────────────────
# ~45-60 minutes on 2x T4 for 200 steps
print('=== STEP 5: GRPO RL Training ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/train_rl_verbalizer.py \
    --data_dir {DATA_DIR} \
    --n_epochs 2 \
    --G 8 \
    --batch_size 4 \
    --beta_kl 0.05 \
    --temperature 0.9
''')
print(out[-5000:])

In [ ]:
# ── 6. Evaluate FVE AFTER RL ──────────────────────────────────────────────────
print('=== STEP 6: Evaluating RL verbalizer FVE (after RL) ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/evaluate_nla.py \
    --data_dir {DATA_DIR} \
    --verbalizer_pt   {DATA_DIR}/verbalizer_rl_best.pt \
    --verbalizer_lora {DATA_DIR}/verbalizer_rl_lora_best
''')
print(out[-3000:])

if os.path.exists(f'{DATA_DIR}/nla_results.json'):
    shutil.copy(f'{DATA_DIR}/nla_results.json', '/kaggle/working/nla_results_rl.json')
    r = json.load(open('/kaggle/working/nla_results_rl.json'))
    print('\n=== RL FVE SUMMARY ===')
    for k, v in r.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')

# Print the comparison
ws = json.load(open('/kaggle/working/nla_results_warmstart.json'))
rl = json.load(open('/kaggle/working/nla_results_rl.json'))
print('\n======= FVE COMPARISON =======')
print(f'  Warm-start generated FVE : {ws.get("fve_generated", ws.get("generated", "?")):.4f}')
print(f'  RL generated FVE         : {rl.get("fve_generated", rl.get("generated", "?")):.4f}')
print(f'  Oracle FVE               : {ws.get("fve_oracle", ws.get("oracle", "?")):.4f}')
print('================================')

In [ ]:
# ── 7. Layer Ablation ─────────────────────────────────────────────────────────
# Train a reconstructor at each of layers {8, 12, 16, 20} and compare FVE
# ~30 mins per layer = ~2 hours total
print('=== STEP 7: Layer Ablation ===')
out = run(f'''
cd /kaggle/working/nla-kth && python src/layer_ablation.py \
    --data_dir {DATA_DIR} \
    --output_dir {DATA_DIR} \
    --layers 8 12 16 20 \
    --n_epochs_recon 5
''')
print(out[-5000:])

In [ ]:
# ── 8. Collect and display all results ───────────────────────────────────────
import json

print('\n' + '='*60)
print('  FINAL RESULTS SUMMARY')
print('='*60)

# RL comparison
print('\n--- RL Phase (GRPO) ---')
ws = json.load(open('/kaggle/working/nla_results_warmstart.json'))
rl = json.load(open('/kaggle/working/nla_results_rl.json'))

def get_fve(d, key):
    for k in [f'fve_{key}', key, f'{key}_fve']:
        if k in d:
            return d[k]
    return None

ws_gen = get_fve(ws, 'generated')
rl_gen = get_fve(rl, 'generated')
oracle = get_fve(ws, 'oracle')
random_fve = get_fve(ws, 'random')

print(f'  Oracle FVE (ceiling)    : {oracle:.4f}')
print(f'  Warm-start generated    : {ws_gen:.4f}')
print(f'  RL generated (GRPO)     : {rl_gen:.4f}  (+{rl_gen - ws_gen:.4f} improvement)')
print(f'  Random FVE (floor)      : {random_fve:.4f}')

gap_closed = (rl_gen - ws_gen) / (oracle - ws_gen) * 100 if oracle != ws_gen else 0
print(f'  Gap closed by RL        : {gap_closed:.1f}%')

# Layer ablation
ablation_path = f'{DATA_DIR}/layer_ablation_summary.json'
if os.path.exists(ablation_path):
    ablation = json.load(open(ablation_path))
    print('\n--- Layer Ablation ---')
    print(f'  {"Layer":<8} {"Oracle FVE":<14} {"Random FVE"}')
    print('  ' + '-'*36)
    for entry in ablation.get('results', ablation.get('layers', [])):
        layer = entry.get('layer', '?')
        o_fve = entry.get('oracle_fve', entry.get('fve_oracle', '?'))
        r_fve = entry.get('random_fve', entry.get('fve_random', '?'))
        marker = ' <-- trained layer' if str(layer) == '16' else ''
        print(f'  {str(layer):<8} {str(round(o_fve,4)):<14} {round(r_fve,4)}{marker}')

print('\n' + '='*60)

# Copy ablation results to working dir for download
if os.path.exists(ablation_path):
    shutil.copy(ablation_path, '/kaggle/working/layer_ablation_summary.json')

# Also copy RL training log
rl_log = f'{DATA_DIR}/rl_training_log.jsonl'
if os.path.exists(rl_log):
    shutil.copy(rl_log, '/kaggle/working/rl_training_log.jsonl')

print('\nAll result files saved to /kaggle/working/ — download them after the run.')
print('Files to grab:')
print('  - nla_results_warmstart.json')
print('  - nla_results_rl.json')
print('  - layer_ablation_summary.json')
print('  - rl_training_log.jsonl')

In [ ]:
# ── 9. Print the exact numbers to paste into README ──────────────────────────
print('\n>>> COPY THESE NUMBERS INTO THE README <<<')
print()
print('Section 4.3 (RL phase) — Results table:')
print(f'  | Warm-start FVE  | {ws_gen:.3f} |')
print(f'  | RL FVE (GRPO)   | {rl_gen:.3f} |')
print(f'  | Oracle FVE      | {oracle:.3f} |')
print(f'  | Gap closed      | {gap_closed:.0f}%  |')
print()

if os.path.exists('/kaggle/working/layer_ablation_summary.json'):
    ablation = json.load(open('/kaggle/working/layer_ablation_summary.json'))
    print('Section 4.4 (Layer ablation) — Results table:')
    print(f'  | Layer | Oracle FVE | Random FVE |')
    print(f'  |-------|------------|------------|')
    for entry in ablation.get('results', ablation.get('layers', [])):
        layer = entry.get('layer', '?')
        o_fve = entry.get('oracle_fve', entry.get('fve_oracle', '?'))
        r_fve = entry.get('random_fve', entry.get('fve_random', '?'))
        print(f'  | {layer:<5} | {round(o_fve,3):<10} | {round(r_fve,3):<10} |')